# 04 · Essentia scores



In [8]:
DRIVE_DIR = "/content/drive/MyDrive/ai-music-validator"
AUDIO_DIR = f"{DRIVE_DIR}/audio"
OUT_CSV = f"{DRIVE_DIR}/essentia_scores.csv"
MODEL_DIR = "/content/essentia_models"
SAVE_EVERY = 10

## 1. Mount Google Drive

In [9]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Install essentia-tensorflow

In [10]:
import subprocess
import sys

print("Python", sys.version.split()[0])
result = subprocess.run([sys.executable, "-m", "pip", "install", "essentia-tensorflow"], capture_output=True, text=True)
print(result.stdout[-1500:])
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError(
        f"pip could not install essentia-tensorflow for Python {sys.version_info.major}.{sys.version_info.minor} "
        "(see the pip error above). 'No matching distribution found' means there is no wheel for this Python "
        "version - check the supported versions at https://pypi.org/project/essentia-tensorflow/#files")

import essentia

print("essentia", essentia.__version__)

Python 3.13.16

essentia 2.1-beta6-dev


## 3. Download the models

From [essentia.upf.edu/models](https://essentia.upf.edu/models/): the `.pb` graph and its `.json` metadata.
Input / output node names and class labels are read from the `.json` files.

In [11]:
import json
import urllib.request
from pathlib import Path

BASE = "https://essentia.upf.edu/models"
MODELS = {  # model name: (folder on the server, column prefix in the CSV; None = the embedding model)
    "discogs-effnet-bs64-1": ("feature-extractors/discogs-effnet", None),
    "genre_discogs400-discogs-effnet-1": ("classification-heads/genre_discogs400", "genre"),
    "mtg_jamendo_instrument-discogs-effnet-1": ("classification-heads/mtg_jamendo_instrument", "instr"),
    "mtg_jamendo_moodtheme-discogs-effnet-1": ("classification-heads/mtg_jamendo_moodtheme", "mood"),
    "voice_instrumental-discogs-effnet-1": ("classification-heads/voice_instrumental", "voice"),
}

Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
meta = {}
for name, (folder, prefix) in MODELS.items():
    for ext in (".pb", ".json"):
        path = Path(MODEL_DIR) / f"{name}{ext}"
        if not path.exists():
            urllib.request.urlretrieve(f"{BASE}/{folder}/{name}{ext}", path)
    meta[name] = json.load(open(Path(MODEL_DIR) / f"{name}.json"))


def node_names(m, purpose):
    """(input node, output node with this output_purpose) from a model's .json metadata."""
    input_name = m["schema"]["inputs"][0]["name"]
    output_name = next(o["name"] for o in m["schema"]["outputs"] if o.get("output_purpose") == purpose)
    return input_name, output_name


for name, (folder, prefix) in MODELS.items():
    purpose = "embeddings" if prefix is None else "predictions"
    print(f"{name}: input / output = {node_names(meta[name], purpose)}, {len(meta[name].get('classes', []))} classes")

discogs-effnet-bs64-1: input / output = ('serving_default_melspectrogram', 'PartitionedCall:1'), 400 classes
genre_discogs400-discogs-effnet-1: input / output = ('serving_default_model_Placeholder', 'PartitionedCall:0'), 400 classes
mtg_jamendo_instrument-discogs-effnet-1: input / output = ('model/Placeholder', 'model/Sigmoid'), 40 classes
mtg_jamendo_moodtheme-discogs-effnet-1: input / output = ('model/Placeholder', 'model/Sigmoid'), 56 classes
voice_instrumental-discogs-effnet-1: input / output = ('model/Placeholder', 'model/Softmax'), 2 classes


## 4. Load the models

In [12]:
from essentia.standard import MonoLoader, TensorflowPredict2D, TensorflowPredictEffnetDiscogs

EMBEDDING_MODEL = "discogs-effnet-bs64-1"
input_name, output_name = node_names(meta[EMBEDDING_MODEL], "embeddings")
embedder = TensorflowPredictEffnetDiscogs(graphFilename=f"{MODEL_DIR}/{EMBEDDING_MODEL}.pb",
                                          input=input_name, output=output_name)
heads = {}
for name, (folder, prefix) in MODELS.items():
    if prefix is None:
        continue
    input_name, output_name = node_names(meta[name], "predictions")
    heads[prefix] = (TensorflowPredict2D(graphFilename=f"{MODEL_DIR}/{name}.pb", input=input_name, output=output_name),
                     meta[name]["classes"])
print({prefix: len(classes) for prefix, (_, classes) in heads.items()})

{'genre': 400, 'instr': 40, 'mood': 56, 'voice': 2}


## 5. Scores for every track (resumable)

In [13]:
import time

import numpy as np
import pandas as pd


def track_scores(path):
    """Mean and max over time of every class activation for one audio file."""
    audio = MonoLoader(filename=str(path), sampleRate=16000, resampleQuality=4)()
    embeddings = embedder(audio)
    row = {}
    for prefix, (model, classes) in heads.items():
        act = np.asarray(model(embeddings))
        for i, label in enumerate(classes):
            row[f"{prefix}__{label}__mean"] = float(act[:, i].mean())
            row[f"{prefix}__{label}__max"] = float(act[:, i].max())
    return row


files = sorted(Path(AUDIO_DIR).rglob("*.mp3"))
print(f"{len(files)} mp3 files under {AUDIO_DIR}")
assert files, "no mp3 files found - check AUDIO_DIR"

done = pd.read_csv(OUT_CSV) if Path(OUT_CSV).exists() else pd.DataFrame()
done_ids = set(done["track_id"]) if len(done) else set()
todo = [f for f in files if f.stem not in done_ids]
print(f"{len(done_ids)} tracks already in {OUT_CSV}, {len(todo)} to go")

rows, failed, t0 = [], [], time.time()
for i, path in enumerate(todo, 1):
    try:
        rows.append({"track_id": path.stem, **track_scores(path)})
    except Exception as e:
        failed.append((path.name, repr(e)))
    if i % SAVE_EVERY == 0 or i == len(todo):
        done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True) if len(done) else pd.DataFrame(rows)
        done.to_csv(OUT_CSV, index=False)
        rows = []
        print(f"[{i}/{len(todo)}] saved {len(done)} tracks, {(time.time() - t0) / i:.1f} s per track", flush=True)

print(f"done: {len(done)} tracks in {OUT_CSV}")
if failed:
    print(f"{len(failed)} failed:", failed)

256 mp3 files under /content/drive/MyDrive/ai-music-validator/audio
46 tracks already in /content/drive/MyDrive/ai-music-validator/essentia_scores.csv, 210 to go
[10/210] saved 56 tracks, 8.5 s per track
[20/210] saved 66 tracks, 7.3 s per track
[30/210] saved 76 tracks, 6.9 s per track
[40/210] saved 86 tracks, 6.8 s per track
[50/210] saved 96 tracks, 6.8 s per track
[60/210] saved 106 tracks, 6.7 s per track
[70/210] saved 116 tracks, 6.7 s per track
[80/210] saved 126 tracks, 6.7 s per track
[90/210] saved 136 tracks, 6.6 s per track
[100/210] saved 146 tracks, 6.6 s per track
[110/210] saved 156 tracks, 6.6 s per track
[120/210] saved 166 tracks, 6.6 s per track
[130/210] saved 176 tracks, 6.6 s per track
[140/210] saved 186 tracks, 6.6 s per track
[150/210] saved 196 tracks, 6.6 s per track
[160/210] saved 206 tracks, 6.6 s per track
[170/210] saved 216 tracks, 6.5 s per track
[180/210] saved 226 tracks, 6.5 s per track
[190/210] saved 236 tracks, 6.5 s per track
[200/210] saved 

## 6. Quick check

In [14]:
scores = pd.read_csv(OUT_CSV).set_index("track_id")
print(f"{scores.shape[0]} tracks, {scores.shape[1]} columns ({scores.shape[1] // 2} labels x mean / max)")
genre_cols = [c for c in scores.columns if c.startswith("genre__") and c.endswith("__mean")]
for tid in scores.index[:5]:
    top = scores.loc[tid, genre_cols].sort_values(ascending=False).head(3)
    print(f"{tid}: voice {scores.loc[tid, 'voice__voice__mean']:.2f} | top genres:",
          ", ".join(f"{c[len('genre__'):-len('__mean')]} ({v:.2f})" for c, v in top.items()))

256 tracks, 996 columns (498 labels x mean / max)
-FFx68qSAuY_1: voice 0.91 | top genres: Rock---Heavy Metal (0.29), Rock---Power Metal (0.23), Rock---Gothic Metal (0.18)
-FFx68qSAuY_4: voice 0.79 | top genres: Rock---Heavy Metal (0.38), Rock---Gothic Metal (0.19), Rock---Power Metal (0.15)
-Q9MTRXS4bE_1: voice 0.87 | top genres: Electronic---House (0.26), Electronic---Dance-pop (0.16), Electronic---Trance (0.11)
-Q9MTRXS4bE_2: voice 0.90 | top genres: Electronic---Tropical House (0.37), Electronic---Dance-pop (0.28), Pop---K-pop (0.24)
-Q9MTRXS4bE_3: voice 0.86 | top genres: Folk, World, & Country---Folk (0.10), Pop---Vocal (0.10), Pop---Ballad (0.07)
